In [26]:
# import things
import numpy as np
import xarray as xr
import pandas as pd
import pickle
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pyproj
from scipy.spatial import cKDTree
from lo_tools import Lfun

import sys
from pathlib import Path

Ldir = Lfun.Lstart()

In [2]:
ds_SSM = xr.open_dataset(
    Ldir['LOu'] / 'intermodel_comparison' / 'ssm_FVCOMICM_00001.nc')

print(ds_SSM.variables)

Frozen({'nprocs': <xarray.Variable (scalar: 1)> Size: 4B
[1 values with dtype=int32]
Attributes:
    long_name:  number of processors, 'partition': <xarray.Variable (nele: 25019)> Size: 100kB
[25019 values with dtype=int32]
Attributes:
    long_name:  partition, 'Initial_Density': <xarray.Variable (siglay: 10, node: 16012)> Size: 640kB
[160120 values with dtype=float32]
Attributes:
    long_name:  Initial Density, 'x': <xarray.Variable (node: 16012)> Size: 64kB
[16012 values with dtype=float32]
Attributes:
    long_name:  nodal x-coordinate
    units:      meters, 'y': <xarray.Variable (node: 16012)> Size: 64kB
[16012 values with dtype=float32]
Attributes:
    long_name:  nodal y-coordinate
    units:      meters, 'h': <xarray.Variable (node: 16012)> Size: 64kB
[16012 values with dtype=float32]
Attributes:
    long_name:      Bathymetry
    units:          meters
    positive:       down
    standard_name:  depth
    grid:           fvcom_grid, 'nv': <xarray.Variable (three: 3, nele: 2

In [3]:
ds_SSM_hydro = xr.open_dataset(
    Ldir['LOu'] / 'intermodel_comparison' / 'ssm_0001.nc',
    decode_times=False)

print(ds_SSM_hydro.variables)

Frozen({'nprocs': <xarray.Variable ()> Size: 4B
[1 values with dtype=int32]
Attributes:
    long_name:  number of processors, 'partition': <xarray.Variable (nele: 25019)> Size: 100kB
[25019 values with dtype=int32]
Attributes:
    long_name:  partition, 'xc': <xarray.Variable (nele: 25019)> Size: 100kB
[25019 values with dtype=float32]
Attributes:
    long_name:  zonal x-coordinate
    units:      meters, 'yc': <xarray.Variable (nele: 25019)> Size: 100kB
[25019 values with dtype=float32]
Attributes:
    long_name:  zonal y-coordinate
    units:      meters, 'siglev': <xarray.Variable (siglev: 11, node: 16012)> Size: 705kB
[176132 values with dtype=float32]
Attributes:
    long_name:      Sigma Levels
    standard_name:  ocean_sigma/general_coordinate
    positive:       up
    valid_min:      -1.0
    valid_max:      0.0
    formula_terms:  sigma:siglay eta: zeta depth: h, 'siglay_center': <xarray.Variable (siglay: 10, nele: 25019)> Size: 1MB
[250190 values with dtype=float32]
Attribut

In [18]:
import requests
import struct

#url = "https://s3.kopah.uw.edu/ssm/hyd/2014/ssm_0001.nc"
url = "https://s3.kopah.uw.edu/ssm/wqm/2014/ssm_FVCOMICM_00001.nc"

# Download the first 1 MB
r = requests.get(
    url,
    headers={"Range": "bytes=0-1048575"}
)

print("Status:", r.status_code)
print("Bytes received:", len(r.content))
print("Content-Range:", r.headers.get("Content-Range"))

# Save temporarily
with open("/tmp/ssm_first_1mb.bin", "wb") as f:
    f.write(r.content)

# NetCDF-3 signature
print("Signature:", r.content[:4])

Status: 206
Bytes received: 1048576
Content-Range: bytes 0-1048575/450089364
Signature: b'CDF\x01'


In [16]:
import fsspec
from scipy.io import netcdf_file

#url = "https://s3.kopah.uw.edu/ssm/hyd/2014/ssm_0001.nc"
url = "https://s3.kopah.uw.edu/ssm/wqm/2014/ssm_FVCOMICM_00001.nc"

# Open the remote file using HTTP range requests
fs = fsspec.filesystem("https")

with fs.open(url, "rb") as f:

    # scipy's NetCDF reader supports NetCDF-3
    ds = netcdf_file(
        f,
        mode="r",
        mmap=False
    )

    print(ds)

In [19]:
print("\nDimensions:")
for name, dim in ds.dimensions.items():
    print(name, dim)

print("\nVariables:")
for name, var in ds.variables.items():
    print(
        f"{name:30s}",
        "dimensions =", var.dimensions,
        "shape =", var.shape
    )


Dimensions:
scalar 1
node 16012
nele 25019
siglay 10
siglev 11
three 3
four 4
nine 9
time None

Variables:
nprocs                         dimensions = ('scalar',) shape = (1,)
partition                      dimensions = ('nele',) shape = (25019,)
Initial_Density                dimensions = ('siglay', 'node') shape = (10, 16012)
x                              dimensions = ('node',) shape = (16012,)
y                              dimensions = ('node',) shape = (16012,)
siglay                         dimensions = ('siglay',) shape = (10,)
siglev                         dimensions = ('siglev',) shape = (11,)
h                              dimensions = ('node',) shape = (16012,)
nv                             dimensions = ('three', 'nele') shape = (3, 25019)
time                           dimensions = ('time',) shape = (24,)
iint                           dimensions = ('time',) shape = (24,)
zeta                           dimensions = ('time', 'node') shape = (24, 16012)
salinity          

In [21]:
var = ds.variables["DOXG"]

print(var)
print(var.shape)

x = var[:1]

print(x)

(24, 10, 16012)
[[[ 7.3361917  9.274833   9.270407  ... 10.924636  10.839997  10.8226385]
  [ 7.332476   9.274413   9.270292  ... 10.924802  10.840011  10.822613 ]
  [ 7.3306713  9.27365    9.00528   ... 10.925752  10.837622  10.916148 ]
  ...
  [ 7.196549   5.489766   5.100958  ... 10.542938  10.665397  10.888554 ]
  [ 7.1367817  5.1716323  4.783319  ... 10.386513  10.461354  10.821949 ]
  [ 7.0500984  4.6988344  4.311016  ... 10.269591  10.364475  10.758964 ]]]


In [25]:
# read pickle with lo ssc obs
data = pd.read_pickle("./combined_bottle_2014_cas7_t1_x11ab_ssc.pkl")
obs = data["obs"]

In [27]:
# Convert SSM node coordinates from UTM 10N to lat/lon

x = ds.variables["x"][:]
y = ds.variables["y"][:]

transformer = pyproj.Transformer.from_crs(
    "epsg:26910",
    "epsg:4326",
    always_xy=True
)

lon_ssm, lat_ssm = transformer.transform(x, y)

print("SSM longitude range:")
print(lon_ssm.min(), lon_ssm.max())

print("\nSSM latitude range:")
print(lat_ssm.min(), lat_ssm.max())

SSM longitude range:
-129.07874552416146 -121.93447807007581

SSM latitude range:
44.39290344817284 52.230207469054974


In [28]:
# Create SSM spatial lookup

ssm_xy = np.column_stack((lon_ssm, lat_ssm))

ssm_tree = cKDTree(ssm_xy)

print("Number of SSM nodes:", len(lon_ssm))

Number of SSM nodes: 16012


In [29]:
# ============================================================
# Select one observation
# ============================================================

row = obs.iloc[0]

obs_lon = row["lon"]
obs_lat = row["lat"]
obs_time = pd.Timestamp(row["time"])
obs_depth = row["z"]

print("Observation:")
print("  lon  =", obs_lon)
print("  lat  =", obs_lat)
print("  time =", obs_time)
print("  z    =", obs_depth)

Observation:
  lon  = -123.24833679199219
  lat  = 48.61833190917969
  time = 2014-02-11 06:35:53
  z    = -1.586479902267456


In [30]:
# ============================================================
# Find nearest SSM node
# ============================================================

distance, node_idx = ssm_tree.query(
    [obs_lon, obs_lat]
)

print("\nNearest SSM node:")
print("  index    =", node_idx)
print("  distance =", distance)

print("\nSSM coordinates:")
print("  lon =", lon_ssm[node_idx])
print("  lat =", lat_ssm[node_idx])


Nearest SSM node:
  index    = 3791
  distance = 0.011983773242016629

SSM coordinates:
  lon = -123.23779374117518
  lat = 48.61263498890764


In [38]:
import numpy as np
import pandas as pd
import pickle
import pyproj
import fsspec

from scipy.io import netcdf_file
from scipy.spatial import cKDTree


# ============================================================
# Settings
# ============================================================

year = 2014

wqm_base = (
    "https://s3.kopah.uw.edu/ssm/wqm/"
    f"{year}/"
)

pickle_path = (
    "combined_bottle_2014_cas7_t1_x11ab_ssc.pkl"
)


# ============================================================
# Load observation targets
# ============================================================

with open(pickle_path, "rb") as f:
    data = pickle.load(f)

obs = data["obs"].copy()


# ============================================================
# Open January 1 file to get SSM horizontal grid
# ============================================================

url_grid = (
    wqm_base +
    "ssm_FVCOMICM_00001.nc"
)

fs = fsspec.filesystem("https")

with fs.open(url_grid, "rb") as f:
    nc_grid = netcdf_file(f, mode="r", mmap=False)

    x = nc_grid.variables["x"][:]
    y = nc_grid.variables["y"][:]

    # SSM x/y are NAD83 / UTM Zone 10N
    transformer = pyproj.Transformer.from_crs(
        "epsg:26910",
        "epsg:4326",
        always_xy=True
    )

    lon_ssm, lat_ssm = transformer.transform(x, y)

    # Nearest-node tree
    ssm_tree = cKDTree(
        np.column_stack((lon_ssm, lat_ssm))
    )

    # Sigma coordinate is constant
    siglay = nc_grid.variables["siglay"][:]


# ============================================================
# Function to extract one observation
# ============================================================

def extract_ssm_observation(row):

    obs_time = pd.Timestamp(row["time"])
    obs_depth = float(row["z"])
    obs_lon = float(row["lon"])
    obs_lat = float(row["lat"])

    # --------------------------------------------------------
    # Determine daily file
    # --------------------------------------------------------

    day_of_year = obs_time.dayofyear

    filename = (
        f"ssm_FVCOMICM_{day_of_year:05d}.nc"
    )

    url = wqm_base + filename

    # --------------------------------------------------------
    # Find nearest SSM node
    # --------------------------------------------------------

    distance, node_idx = ssm_tree.query(
        [obs_lon, obs_lat]
    )

    # --------------------------------------------------------
    # Open daily file remotely
    # --------------------------------------------------------

    with fs.open(url, "rb") as f:

        nc = netcdf_file(
            f,
            mode="r",
            mmap=False
        )

        # ----------------------------------------------------
        # SSM hourly times
        # ----------------------------------------------------

        time_values = nc.variables["time"][:]

        # Seconds since beginning of day
        seconds_since_midnight = (
            obs_time.hour * 3600
            + obs_time.minute * 60
            + obs_time.second
            + obs_time.microsecond / 1e6
        )

        time_idx = np.argmin(
            np.abs(
                time_values -
                seconds_since_midnight
            )
        )

        # ----------------------------------------------------
        # Water-column depth from SSM
        # ----------------------------------------------------

        H = float(
            nc.variables["depth"][
                time_idx,
                node_idx
            ]
        )

        # ----------------------------------------------------
        # Convert sigma coordinate to physical depth
        #
        # siglay is negative, surface = 0
        # ----------------------------------------------------

        layer_depths = siglay * H

        # ----------------------------------------------------
        # Find closest SSM layer to observation depth
        # ----------------------------------------------------

        layer_idx = np.argmin(
            np.abs(
                layer_depths -
                obs_depth
            )
        )

        # ----------------------------------------------------
        # Extract variables
        # ----------------------------------------------------

        DO = float(
            nc.variables["DOXG"][
                time_idx,
                layer_idx,
                node_idx
            ]
        )

        temp = float(
            nc.variables["temp"][
                time_idx,
                layer_idx,
                node_idx
            ]
        )

        salinity = float(
            nc.variables["salinity"][
                time_idx,
                layer_idx,
                node_idx
            ]
        )

        # ----------------------------------------------------
        # Return results
        # ----------------------------------------------------

        return {
            "ssm_node": node_idx,
            "ssm_distance": distance,
            "ssm_time_idx": time_idx,
            "ssm_H": H,
            "ssm_layer": layer_idx,
            "ssm_depth": layer_depths[layer_idx],
            "ssm_DOXG": DO,
            "ssm_temp": temp,
            "ssm_salinity": salinity,
            "ssm_file": filename,
        }

In [39]:
result = extract_ssm_observation(obs.iloc[0])

for key, value in result.items():
    print(f"{key}: {value}")

ssm_node: 3791
ssm_distance: 0.011983773242016629
ssm_time_idx: 0
ssm_H: 177.4397430419922
ssm_layer: 0
ssm_depth: -2.8055684566497803
ssm_DOXG: 9.522340774536133
ssm_temp: 7.006908416748047
ssm_salinity: 30.17892837524414
ssm_file: ssm_FVCOMICM_00042.nc


In [49]:
print("time units:")
print(ds.variables["time"].units)

print("\ntime long_name:")
print(ds.variables["time"].long_name if hasattr(ds.variables["time"], "long_name") else "none")

print("\ntime calendar:")
print(ds.variables["time"].calendar if hasattr(ds.variables["time"], "calendar") else "none")

time units:
b'seconds after 00:00:00'

time long_name:
b'Time'

time calendar:
b'none'
